# Lab 17 · Decision models: a weighted decision matrix with a scoring model

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Steps 4–5 use TypeSafe and require `TYPESAFE_API_KEY` in Colab Secrets.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
if not find_spec('typesafe_sdk'):
    %pip install -q 'typesafe-sdk==0.7.2'

## Warm-up · Give important criteria more weight

Two activities have illustrative scores out of five for cost and shelter from rain. Give shelter a higher weight, then compare the totals. These numbers are teaching choices, not measured facts.

In [ ]:
walk = {"cost": 5, "shelter": 1}
museum = {"cost": 2, "shelter": 5}
weights = {"cost": 1, "shelter": 3}
walk_score = walk["cost"] * weights["cost"] + walk["shelter"] * weights["shelter"]
museum_score = museum["cost"] * weights["cost"] + museum["shelter"] * weights["shelter"]
print("Walk:", walk_score, "| Museum:", museum_score)

**Next:** Use the same idea to compare a task queue and an event log.

## Make a communication trade-off explicit
**Start with a problem:** A simple average treats replay and simplicity as equally important even when the task has different priorities.

**In this lab:** Compare the options with different weights, then consider how a hard requirement limits the choices.

**So far:** agents reviewed microservice boundaries, resilience, and contracts. **New:** a weighted matrix exposes the priorities behind a recommendation.

ShopEasy must choose between a **task queue** (workers compete for deliveries) and a **retained event log** such as Kafka (subscriber groups independently read and replay retained records). Compare Task distribution, Replay, Simplicity, and Fan-out.

The 0–4 scores and weights are **illustrative judgements**, not measurements or universal product rankings. Keep hard constraints separate: if replay is mandatory, a high simplicity score cannot make a non-replayable task queue eligible. A TypeSafe model supplies scores; Python controls the decision.

**At a glance**

![Lab 17 at a glance: Two options (task queue · event log), then Score criteria (0–4 per option) or JEV scores (Score primitive), then Weights (the team’s priorities), then Weighted total (Σ score × weight), then Winner (higher total wins). Then back to Weighted total: reweight, never re-judge.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/17.svg)

## Think about the example

A task queue distributes work among competing workers. A retained event log lets separate subscriber groups read events independently. A record key groups related records; eventId identifies a fact for deduplication.

**Concepts:** task queues versus retained event logs; independent consumer groups and replay; partition key versus eventId; consumer parallelism is capped by partitions.

Use the scenario above to consider these questions before running the code:

1. Should competing workers process a task, or should each subscriber group read the same event?
2. Why do orderId as the Kafka key and eventId for deduplication solve different problems?

## The idea in context

A weighted matrix exposes the assumptions behind a choice. Code combines the scores; a scoring model supplies judgements rather than implementing broker guarantees.

A task queue distributes deliveries among workers. A retained event log lets subscriber groups read and replay independently. Kafka groups are limited by partition count; eventId supports deduplication while a record key determines grouping under a stable partition mapping.

| Key term | Meaning |
| --- | --- |
| **Decision model** | A repeatable procedure that turns judgements into a choice. |
| **Weighted decision matrix** | Option scores per criterion, each multiplied by the criterion's weight and summed. |
| **Weight** | How much one criterion counts toward the choice; the team's stated priority. |
| **System One model** | A model such as TypeSafe JEV that returns typed judgements and probabilities, not prose. |

## Main exercise at a glance

1. Compare a task queue and event log
2. Weight a simple work-distribution task
3. Explore replay and subscriber priorities
4. Give the scoring model the evidence
5. Score once, reweight in code

## Step 1 · Compare a task queue and event log

A plain average weights every criterion equally. A task queue distributes work among competing workers; an event log can retain events for independent subscribers. The scores are teaching fixtures. The record key keeps one order's events together under a stable partition mapping; eventId identifies the fact for deduplication. Neither is an architecture score.

In [ ]:
criteria = ["Task distribution", "Replay", "Simplicity", "Fan-out"]
queue = [4, 0, 4, 1]  # competing workers; this task-queue example has no replay requirement
log = [2, 4, 1, 4]    # retained records; independent subscriber groups; more concepts to run
record = {"key": "order-17", "eventId": "evt-9001", "eventType": "OrderPlaced"}
print("Partition grouping:", record["key"], "| deduplication:", record["eventId"])

def average(scores):
    return sum(scores) / len(scores)

for name, scores in [("Task queue", queue), ("Event log", log)]:
    print(f"{name:<17} average {average(scores):.2f}")

## Step 2 · Weight a simple work-distribution task

For a small background-work queue where each delivery goes to one worker and replay is optional, the team gives Simplicity weight 3. This is a scenario preference, not a universal comparison of every queue and log product.

`zip(scores, weights)` pairs values in the same position: the first score with the first weight, and so on. Divide the weighted sum by the sum of weights to keep the 0–4 scale.

In [ ]:
weights = [1, 1, 3, 1]

def weighted(scores):
    total = 0
    for score, weight in zip(scores, weights):
        total += score * weight
    return total / sum(weights)

table = [(name, weighted(s)) for name, s in [("Task queue", queue), ("Event log", log)]]
for name, score in table:
    print(f"{name:<17} {score:.2f}")
print("winner:", max(table, key=lambda r: r[1])[0])

## Step 3 · Explore replay and subscriber priorities

Keep the same facts and scores, but give Replay weight 3 and Fan-out weight 2. The winner flips. Reweighting needs no new model call; changing the real workload requires reassessing scores too.

The last calculation recalls Kafka consumer groups: with four partitions and five consumers in one group, only four can own a partition at once. Adding an independent subscriber group gives that group its own progress; it does not raise this group's partition limit.

In [ ]:
weights_2 = [1, 3, 0, 2]

def weighted_2(scores):
    total = 0
    for score, weight in zip(scores, weights_2):
        total += score * weight
    return total / sum(weights_2)

table2 = [(name, weighted_2(s)) for name, s in [("Task queue", queue), ("Event log", log)]]
for name, score in table2:
    print(f"{name:<17} {score:.2f}")
print("winner:", max(table2, key=lambda r: r[1])[0])
partitions, consumers = 4, 5
print("Active consumers in this group:", min(partitions, consumers))

## Step 4 · Give the scoring model the evidence

All eight judgements use the same ordered rubric. The state supplies the queue and event-log descriptions and what each criterion means. A score remains a judgement, not a benchmark result. This step needs a separate `TYPESAFE_API_KEY` in Colab Secrets; a FreeInference key does not work with TypeSafe.

In [ ]:
from typesafe_sdk import Score, TypeSafeClient
from coursekit import secret
state = {
    "case": "ShopEasy background work; replay and multiple subscribers are preferences, not hard constraints.",
    "options": {"Task queue": "Competing workers handle tasks; no replay in this example.",
                "Event log": "Retained records; independent subscriber groups can replay; operate partitions and offsets."},
    "criteria": {"Task distribution": "One worker per delivery", "Replay": "Re-read retained history",
                 "Simplicity": "Few moving parts for a small workload", "Fan-out": "Independent readers receive each fact"},
}
rubric = ["fails the criterion", "weak support", "partly supports", "supports well", "strong support"]
print("built", len(state["options"]), "options and", len(criteria), "criteria")

## Step 5 · Score once, reweight in code

One request asks independent option–criterion questions together. The cell reads a key, makes this request, and combines the returned scores with the Python matrix. Typed output guarantees a shape, not the correctness of an architecture recommendation.

In [ ]:
def ask_scores():
    questions = {}
    for option in state["options"]:
        for criterion in criteria:
            question_id = f"{option}:{criterion}"
            question = f"How well does {option} support {criterion}, as defined in state?"
            questions[question_id] = Score(instructions=question, criteria=rubric)
    with TypeSafeClient(api_key=secret("TYPESAFE_API_KEY")) as client:
        return client.system_one(state=state, questions=questions)

resp = ask_scores()
scored = {o: [resp.scores[f"{o}:{c}"].score for c in criteria] for o in state["options"]}
for name, scores in scored.items():
    print(f"{name:<17} {weighted_2(scores):.2f}")

## What you should see

The averages are 2.25 for the task queue and 2.75 for the event log. Simplicity-weighted totals are 2.83 and 2.17: the queue wins. Replay/fan-out weights give 1.00 and 3.67: the log wins. Four partitions allow at most four active consumers in this five-consumer group. The TypeSafe scores are separate model judgements and may differ.

## Try it

1. Make replay a hard requirement. Which option must be filtered out before scoring?
2. In `record`, change the key to `customerId`. Whose events must now stay together? Why is that different from `eventId`?
3. Change the partition and consumer counts. Does adding a fifth consumer to four partitions help this group's parallelism?
4. Inspect `resp.scores[...].confidence` and check the reasoning against the queue and event-log descriptions in Step 1. A concentrated judgement is not a correctness guarantee.
5. For RideNow, choose the event key, subscriber groups, and deduplication ID before comparing broker choices.

## Takeaway

Expose priorities, keep hard constraints separate, and retain the example's communication semantics. A partition key groups records; eventId identifies a fact; group offsets track each subscriber's progress. A scoring model does not implement those guarantees.

## Read further

- [TypeSafe Score documentation](https://docs.typesafe.ai/primitives/score)
- [TypeSafe Python SDK](https://docs.typesafe.ai/sdk/python)